# CS4100 Final Project  
Team Members: Khushi Khan, Dustin Zhang, Kayla Handley, Koena Gupta

In [26]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torchvision
import torchmetrics
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split    
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from tqdm import tqdm

In [27]:
mp3_df = pd.read_csv('../data/cleaned/fma_cleaned_dataset_emotion_labels.csv', low_memory=False)

mp3_df.head()

,track_id,title,genre_top,mp3_path,valence,energy,emotion_joy_excitement,emotion_peaceful_content,emotion_anger_tension,emotion_sadness,emotion_joy_excitement_softmax,emotion_peaceful_content_softmax,emotion_anger_tension_softmax,emotion_sadness_softmax
0,2,Food,Hip-Hop,../data/raw/fma_small/000/000002.mp3,0.576661,0.634476,0.606258,0.482776,0.539340,0.395489,0.275553,0.243544,0.257717,0.223187
1,5,This World,Hip-Hop,../data/raw/fma_small/000/000005.mp3,0.621661,0.701470,0.662768,0.487639,0.563560,0.340779,0.288217,0.241914,0.260996,0.208873
2,10,Freeway,Pop,../data/raw/fma_small/000/000010.mp3,0.963590,0.924525,0.944260,0.683448,0.654245,0.059254,0.341135,0.262819,0.255255,0.140791
3,140,Queen Of The Wires,Folk,../data/raw/fma_small/000/000140.mp3,0.609991,0.265685,0.470467,0.675022,0.333688,0.587931,0.236755,0.290493,0.206489,0.266264
4,141,Ohio,Folk,../data/raw/fma_small/000/000141.mp3,0.163950,0.075632,0.127671,0.663828,0.593591,0.881316,0.155578,0.265949,0.247911,0.330562


In [28]:
# Targets to predict
y = mp3_df[['valence', 'energy']].to_numpy()
n_outputs = y.shape[1]

# Features to pass in
X = mp3_df.drop(columns=['valence', 'energy']).to_numpy()
n_samples, n_features = X.shape

In [29]:
# Splitting data into train and test sets
X_train, y_train, X_test, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

emotion_classes = ('emotion_joy_excitement', 'emotion_peaceful_content', 'emotion_anger_tension', 'emotion_sadness')

In [33]:
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(CNN, self).__init__()

        # 1st convolutional layer
        self.conv1 = nn.Conv2d(
            in_channels=in_channels, 
            out_channels=6,
            kernel_size=3,
            padding=1)
        
        # Max pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # 2nd convolutional layer
        self.conv2 = nn.Conv2d(
            in_channels=6, 
            out_channels=16, 
            kernel_size=3,
            padding=1)

        # Fully connected layer
        self.fc1 = nn.Linear(16 * 5 * 5, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))  # Apply first convolution and ReLU activation
        x = self.pool(x)           # Apply max pooling
        x = F.relu(self.conv2(x))  # Apply second convolution and ReLU activation
        x = self.pool(x)           # Apply max pooling
        x = x.reshape(x.shape[0], -1)  # Flatten the tensor
        x = self.fc1(x)            # Apply fully connected layer
        return x

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CNN(in_channels=3, num_classes=len(emotion_classes)).to(device)

print(model)

CNN(
  (conv1): Conv2d(3, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=400, out_features=4, bias=True)
)


In [34]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [35]:
num_epochs=10
for epoch in range(num_epochs):
 # Iterate over training batches
   print(f"Epoch [{epoch + 1}/{num_epochs}]")

   for batch_index, (data, targets) in enumerate(tqdm(X_train)):
       data = data.to(device)
       targets = targets.to(device)
       scores = model(data)
       loss = criterion(scores, targets)
       optimizer.zero_grad()
       loss.backward()
       optimizer.step()

Epoch [1/10]


  0%|          | 0/905 [00:00<?, ?it/s]


ValueError: too many values to unpack (expected 2)